In [1]:
# Cell 1: imports and global settings
import numpy as np
import pandas as pd
from datasets import load_dataset

RANDOM_STATE = 42          # one seed reused everywhere for reproducibility
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)   # show all columns when printing

/home/doomguy/Projects/24127441/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: download the dataset from Hugging Face and inspect its structure
raw_dataset = load_dataset("cestwc/census-income")
print(raw_dataset)

DatasetDict({
    train: Dataset({
        features: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'],
        num_rows: 32561
    })
    test: Dataset({
        features: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'],
        num_rows: 16281
    })
})


In [3]:
# Cell 3: convert each split to a pandas DataFrame
# split is a Dataset object, which is not as convenient to work with as a DataFrame
# We convert each split to a pandas DataFrame (2D table)
# name is split name (train, test, validation)
split_frames = {name: split.to_pandas() for name, split in raw_dataset.items()}

# split_frames is a dictionary (key, value pairs)
for name, frame in split_frames.items():
    print(f"{name}: {frame.shape[0]} rows, {frame.shape[1]} columns")

# iter() makes all values a list to traverse, next() gets the first value in that list
first_frame = next(iter(split_frames.values()))
print(first_frame.columns.tolist())
first_frame.head()

train: 32561 rows, 15 columns
test: 16281 rows, 15 columns
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,7,77516,9,13,4,1,1,4,1,2174,0,40,39,0
1,50,6,83311,9,13,2,4,0,4,1,0,0,13,39,0
2,38,4,215646,11,9,0,6,1,4,1,0,0,40,39,0
3,53,4,234721,1,7,2,6,0,2,1,0,0,40,39,0
4,28,4,338409,9,13,2,10,5,2,0,0,0,40,5,0


In [ ]:
# Cell 4: inspect dtypes and label values in each split
train_frame = split_frames["train"]
test_frame = split_frames["test"]

print(train_frame.dtypes)
print()
print("train income values:", train_frame["income"].unique())
print("test income values:", test_frame["income"].unique())

# from table in cell 3 output, we see that the dataset is label-encoded - text fields are replaced with integer
# that why you see dtypes of each column is int64 in this output cell
# we decode those columns with encoding ones in cell 8

age               int64
workclass         int64
fnlwgt            int64
education         int64
education-num     int64
marital-status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country    int64
income            int64
dtype: object

train income values: [0 1]
test income values: [0 1]


In [5]:
# Cell 5: how are missing values encoded?
print("workclass values:", train_frame["workclass"].unique())

# count real NaN values across the whole table
total_real_nan = int(train_frame.isna().sum().sum())
print("real NaN count:", total_real_nan)

# count '?' placeholders per column (True counts as 1 when summed)
question_mark_counts = (train_frame == "?").sum()
print(question_mark_counts[question_mark_counts > 0])

workclass values: [7 6 4 1 2 0 5 8 3]
real NaN count: 0
Series([], dtype: int64)


In [6]:
# Cell 6: look at the feature types stored in the dataset metadata
train_features = raw_dataset["train"].features
print(train_features["workclass"])
print(train_features["income"])

ClassLabel(names=['?', 'Federal-gov', 'Local-gov', 'Never-worked', 'Private', 'Self-emp-inc', 'Self-emp-not-inc', 'State-gov', 'Without-pay'])
ClassLabel(names=['<=50K', '>50K'])


In [7]:
# Cell 7: count how often each workclass code appears
workclass_code_counts = train_frame["workclass"].value_counts().sort_index()
print(workclass_code_counts)

workclass
0     1836
1      960
2     2093
3        7
4    22696
5     1116
6     2541
7     1298
8       14
Name: count, dtype: int64


In [8]:
# Cell 8: decode ClassLabel columns back to text, turn "?" into NaN
from datasets import ClassLabel

def decode_frame(frame, features):
    """Return a copy of frame where integer class codes are replaced by their text names."""
    decoded = frame.copy()                      # never modify the original frame
    for column_name, feature in features.items():
        if isinstance(feature, ClassLabel):     # only categorical columns have names
            code_to_name = dict(enumerate(feature.names))   # {0: '?', 1: 'Federal-gov', ...}
            decoded[column_name] = frame[column_name].map(code_to_name)
    return decoded

train_decoded = decode_frame(train_frame, raw_dataset["train"].features)
test_decoded = decode_frame(test_frame, raw_dataset["test"].features)

train_decoded = train_decoded.replace("?", np.nan)
test_decoded = test_decoded.replace("?", np.nan)

train_decoded.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [9]:
# Cell 9: missing values per column, in count and percent
missing_count = train_decoded.isna().sum()
missing_percent = (train_decoded.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({"missing_count": missing_count, "missing_percent": missing_percent})
print(missing_summary[missing_summary["missing_count"] > 0])
print(train_decoded.dtypes)

                missing_count  missing_percent
workclass                1836             5.64
occupation               1843             5.66
native-country            583             1.79
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object


In [10]:
# Cell 10: merge the two official splits into one table
full_data = pd.concat([train_decoded, test_decoded], ignore_index=True)
print("merged shape:", full_data.shape)
print(full_data["income"].value_counts(normalize=True).round(4))

merged shape: (48842, 15)
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64


In [11]:
# Cell 11: stratified 80/20 split (done BEFORE any analysis that learns from data)
from sklearn.model_selection import train_test_split

features = full_data.drop(columns=["income"])   # the 14 input columns
target = full_data["income"]                    # the label

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    stratify=target,             # keep the <=50K / >50K ratio in both parts
    random_state=RANDOM_STATE,   # same split on every run
)

print("train:", X_train.shape, "test:", X_test.shape)
print("train income share:")
print(y_train.value_counts(normalize=True).round(4))
print("test income share:")
print(y_test.value_counts(normalize=True).round(4))

train: (39073, 14) test: (9769, 14)
train income share:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64
test income share:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64


In [12]:
# Cell 12: D'Agostino-Pearson test on each numeric training column
from scipy import stats

ALPHA = 0.05   # significance level

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
print("numeric columns:", numeric_columns)

normality_rows = []   # will hold one dict per column
for column_name in numeric_columns:
    values = X_train[column_name].dropna()
    statistic, p_value = stats.normaltest(values)
    normality_rows.append({
        "column": column_name,
        "n": len(values),
        "statistic": statistic,
        "p_value": p_value,
        "skewness": stats.skew(values),
        "is_normal": p_value >= ALPHA,
    })

normality_results = pd.DataFrame(normality_rows).set_index("column")
print(normality_results)

numeric columns: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
                    n     statistic        p_value   skewness  is_normal
column                                                                  
age             39073   1851.163896   0.000000e+00   0.559983      False
fnlwgt          39073  12539.369301   0.000000e+00   1.424819      False
education-num   39073   1051.850376  3.922771e-229  -0.327093      False
capital-gain    39073  66917.958926   0.000000e+00  12.233423      False
capital-loss    39073  35187.884948   0.000000e+00   4.595461      False
hours-per-week  39073   3026.330895   0.000000e+00   0.231421      False
